# 02 — Q4 2025 earnings-event dataset (WRDS Capital IQ)

Built to run **inside Austin's branch**, using the same `ciq_transcripts` tables as `notebooks/ciq_transcripts_eda.ipynb`.

**Purpose:** create one research-ready record per earnings-call event, with full ordered transcript text, component-level speaker metadata, a *first-available* transcript version and (optionally) a final/corrected version. This is not yet an EPS/returns merge.

**Important:** Capital IQ's call date is **not necessarily the earnings release timestamp**. Transcript creation timestamp is a proxy for availability, not proof of actual point-in-time availability. No intraday alpha or event-day return inference should rely on these alone. Confirm permitted use of WRDS data for the NTAM collaboration before exporting/sharing full text.

## 0. Setup
Run from the repository root, or set `PROJECT_ROOT` explicitly. Dependencies: `wrds`, `pandas`, `pyarrow` (for parquet). No PDFs needed.

In [12]:
# If needed, uncomment:
# %pip install wrds pandas pyarrow
from pathlib import Path
import json, re, time
import pandas as pd
import wrds
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUT = PROJECT_ROOT / 'data' / 'processed' / 'earnings_events_q4_2025'
OUT.mkdir(parents=True, exist_ok=True)

START_DATE = '2025-10-01'
END_DATE = '2025-12-31'
US_ONLY = True
PILOT_ONLY = True     # True: first try five pilot firms; False: all eligible US firms
PILOT_COMPANIES = ['Microsoft', 'JPMorgan', 'Coca-Cola', 'Caterpillar', 'Oracle']
MAX_EVENTS = 50 if PILOT_ONLY else None   # Set None to retrieve every eligible event
BATCH_SIZE = 25
FETCH_FINAL_VERSION = True
MIN_WORDS = 1000
print('Output:', OUT.resolve())

Output: C:\Users\vidhi\OneDrive\Desktop\Project lab\Autumn 26\LLM-based-earnings-surprise_NTAM\data\processed\earnings_events_q4_2025


## 1. Connect and verify dataset access
If the connection fails or `ciq_transcripts` is absent, ask Austin about the authorized access method; do not use his credentials.

In [14]:
import wrds

db = wrds.Connection(
    wrds_username="YOUR_ACTUAL_WRDS_USERNAME"
)

print("Connected successfully!")

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\vidhi\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected successfully!


In [15]:

libs = db.list_libraries()
assert 'ciq_transcripts' in libs, 'ciq_transcripts not visible in your WRDS account.'
print('Capital IQ transcripts library available')

Capital IQ transcripts library available


## 2. Fetch Q4 2025 earnings calls and transcript versions

Use **event type 48** as in Austin's EDA. Keep all transcript versions at this stage; filter U.S. incorporation using the CIQ geography join. Pilot company matching is a **reviewable name filter**, not a definitive ticker mapping.

In [16]:
sql = """
SELECT d.keydevid, d.transcriptid, d.companyid, d.companyname,
       d.mostimportantdateutc AS call_date,
       d.mostimportanttimeutc AS call_time_utc,
       d.transcriptcreationdate_utc AS created_date,
       d.transcriptcreationtime_utc AS created_time_utc,
       d.transcriptpresentationtypeid,
       d.transcriptpresentationtypename AS presentation,
       d.transcriptcollectiontypename AS collection,
       d.audiolengthsec,
       g.isocountry2 AS country_code
FROM ciq_transcripts.wrds_transcript_detail d
LEFT JOIN ciq.ciqcompany c ON c.companyid = d.companyid
LEFT JOIN ciq.ciqcountrygeo g ON g.countryid = c.countryid
WHERE d.keydeveventtypeid = 48
  AND d.mostimportantdateutc BETWEEN %(start)s AND %(end)s
ORDER BY d.keydevid, d.transcriptcreationdate_utc, d.transcriptcreationtime_utc, d.transcriptid
"""
versions = db.raw_sql(sql, params={'start': START_DATE, 'end': END_DATE},
                      date_cols=['call_date','created_date'])
print('Versions:', len(versions), 'Events:', versions.keydevid.nunique())
if US_ONLY:
    versions = versions[versions.country_code.eq('US')].copy()
if PILOT_ONLY:
    pattern = '|'.join(re.escape(x) for x in PILOT_COMPANIES)
    versions = versions[versions.companyname.str.contains(pattern, case=False, na=False)].copy()
print('Filtered versions:', len(versions), 'Events:', versions.keydevid.nunique())
display(versions[['companyname','call_date','transcriptid','presentation','collection']].head(20))
if versions.empty:
    raise ValueError('No calls matched. Inspect available company names and toggle PILOT_ONLY=False.')

Versions: 18573 Events: 7138
Filtered versions: 16 Events: 5


,companyname,call_date,transcriptid,presentation,collection
38,JPMorgan Chase & Co.,2025-10-14,3560280.0,Preliminary,Spellchecked Copy
39,JPMorgan Chase & Co.,2025-10-14,3560303.0,Final,Edited Copy
40,JPMorgan Chase & Co.,2025-10-14,3616842.0,Final,Proofed Copy
41,JPMorgan Chase & Co.,2025-10-14,3617520.0,Final,Proofed Copy
1370,Caterpillar Inc.,2025-10-29,3570518.0,Preliminary,Spellchecked Copy
1371,Caterpillar Inc.,2025-10-29,3570692.0,Final,Edited Copy
1372,Caterpillar Inc.,2025-10-29,3572025.0,Final,Proofed Copy
2409,The Coca-Cola Company,2025-10-21,3562705.0,Preliminary,Spellchecked Copy
2410,The Coca-Cola Company,2025-10-21,3562851.0,Final,Edited Copy
2411,The Coca-Cola Company,2025-10-21,3619015.0,Final,Proofed Copy


## 3. Deduplicate at the event level, retaining version metadata

- **First version:** earliest recorded creation date/time (prefer for point-in-time research, subject to availability checks).
- **Final version:** latest version flagged with `transcriptpresentationtypeid = 5` (Austin's EDA criterion); otherwise latest recorded version. For descriptive analysis only unless timing verified.
- Preserve `keydevid`, `companyid`, transcript IDs, and creation times.


In [17]:
v = versions.copy()
v['created_at_utc'] = pd.to_datetime(
    v['created_date'].dt.strftime('%Y-%m-%d') + ' ' + v['created_time_utc'].astype(str),
    errors='coerce', utc=True)
v['created_sort'] = v['created_at_utc'].fillna(pd.Timestamp('1900-01-01',tz='UTC'))
v = v.sort_values(['keydevid','created_sort','transcriptid'])
first = v.drop_duplicates('keydevid', keep='first').copy()
final_candidates = v.sort_values(['keydevid','transcriptpresentationtypeid','created_sort','transcriptid'])
# Explicitly prioritize presentation type 5, then most recent creation
final_candidates['is_final'] = final_candidates.transcriptpresentationtypeid.eq(5).astype(int)
final = final_candidates.sort_values(['keydevid','is_final','created_sort','transcriptid']).drop_duplicates('keydevid', keep='last')

meta = first[['keydevid','companyid','companyname','country_code','call_date','call_time_utc','audiolengthsec']].copy()
meta = meta.merge(first[['keydevid','transcriptid','created_at_utc','presentation','collection']].rename(columns={
    'transcriptid':'first_transcriptid','created_at_utc':'first_created_at_utc',
    'presentation':'first_presentation','collection':'first_collection'}), on='keydevid')
meta = meta.merge(final[['keydevid','transcriptid','created_at_utc','presentation','collection']].rename(columns={
    'transcriptid':'final_transcriptid','created_at_utc':'final_created_at_utc',
    'presentation':'final_presentation','collection':'final_collection'}), on='keydevid')
meta['n_versions'] = meta.keydevid.map(v.groupby('keydevid').size())
meta = meta.sort_values(['call_date','companyname','keydevid'])
if MAX_EVENTS is not None:
    meta = meta.head(MAX_EVENTS).copy()
print('Selected events:',len(meta))
display(meta[['companyname','call_date','n_versions','first_transcriptid','final_transcriptid']].head(20))

Selected events: 5


,companyname,call_date,n_versions,first_transcriptid,final_transcriptid
0,JPMorgan Chase & Co.,2025-10-14,4,3560280.0,3617520.0
2,The Coca-Cola Company,2025-10-21,3,3562705.0,3619015.0
1,Caterpillar Inc.,2025-10-29,3,3570518.0,3572025.0
3,Microsoft Corporation,2025-10-29,3,3571533.0,3576017.0
4,Oracle Corporation,2025-12-10,3,3617991.0,3618101.0


## 4. Pull ordered transcript components in small batches

Austin's EDA joins `ciqtranscriptcomponent` to `ciqtranscriptperson`, `ciqtranscriptcomponenttype` and `ciqtranscriptspeakertype`. This step downloads only selected event IDs, not the entire WRDS corpus. Keep source text unchanged in the structured output.

In [18]:
ids = set(meta.first_transcriptid.dropna().astype(int))
if FETCH_FINAL_VERSION:
    ids.update(meta.final_transcriptid.dropna().astype(int))
ids = sorted(ids)
print('Unique transcript versions to download:',len(ids))

def fetch_components(id_batch):
    if not id_batch: return pd.DataFrame()
    safe_ids = ','.join(str(int(x)) for x in id_batch)
    query = f"""
    SELECT c.transcriptid, c.componentorder,
           ct.transcriptcomponenttypename AS component_type,
           p.transcriptpersonname AS speaker,
           st.speakertypename AS speaker_type,
           p.companyname AS speaker_company,
           c.componenttext
    FROM ciq_transcripts.ciqtranscriptcomponent c
    LEFT JOIN ciq_transcripts.ciqtranscriptcomponenttype ct
      USING (transcriptcomponenttypeid)
    LEFT JOIN ciq_transcripts.ciqtranscriptperson p
      USING (transcriptpersonid)
    LEFT JOIN ciq_transcripts.ciqtranscriptspeakertype st
      ON st.speakertypeid = p.speakertypeid
    WHERE c.transcriptid IN ({safe_ids})
    ORDER BY c.transcriptid, c.componentorder
    """
    return db.raw_sql(query)

parts = []
for start in range(0, len(ids), BATCH_SIZE):
    batch = ids[start:start+BATCH_SIZE]
    df = fetch_components(batch)
    parts.append(df)
    print(f'Batch {start//BATCH_SIZE+1}: {len(batch)} transcripts, {len(df)} components')
components = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
assert not components.empty, 'No transcript text returned; verify CIQ table access.'
components = components.sort_values(['transcriptid','componentorder'],kind='stable')
print('Total components:',len(components))
display(components.drop(columns='componenttext').head(8))

Unique transcript versions to download: 10
Batch 1: 10 transcripts, 446 components
Total components: 446


,transcriptid,componentorder,component_type,speaker,speaker_type,speaker_company
0,3560280,0,Presentation Operator Message,Operator,Operator,<NA>
1,3560280,1,Presenter Speech,Jeremy Barnum,Executives,<NA>
2,3560280,2,Question and Answer Operator Message,Operator,Operator,<NA>
3,3560280,3,Question,John McDonald,Analysts,<NA>
4,3560280,4,Answer,Jeremy Barnum,Executives,<NA>
5,3560280,5,Question,John McDonald,Analysts,<NA>
6,3560280,6,Answer,Jeremy Barnum,Executives,<NA>
7,3560280,7,Question and Answer Operator Message,Operator,Operator,<NA>


## 5. Build structured and plain-text transcript versions

Preserve speakers, component type, ordering, and exact source text. The LLM notebook can use the rendered text or structured turns. We do not silently delete disclaimers or operator remarks.

In [19]:
def safe_str(x):
    return '' if pd.isna(x) else str(x)

def build_transcript(g):
    turns = []
    for row in g.itertuples(index=False):
        raw = safe_str(row.componenttext)
        if not raw.strip(): continue
        turns.append({
            'order': int(row.componentorder) if pd.notna(row.componentorder) else None,
            'component_type': safe_str(row.component_type),
            'speaker': safe_str(row.speaker),
            'speaker_type': safe_str(row.speaker_type),
            'speaker_company': safe_str(row.speaker_company),
            'text': raw,
        })
    text = '\n\n'.join(
        f"[{t['component_type']}] {t['speaker']} ({t['speaker_type']})\n{t['text']}"
        for t in turns)
    word_count = sum(len(t['text'].split()) for t in turns)
    return {'turns':turns,'text':text,'word_count':word_count,'n_components':len(turns)}

transcript_map = {int(tid):build_transcript(g) for tid,g in components.groupby('transcriptid',sort=False)}
rows = []
for r in meta.itertuples(index=False):
    first_id = int(r.first_transcriptid)
    final_id = int(r.final_transcriptid)
    ft = transcript_map.get(first_id, {'turns':[],'text':'','word_count':0,'n_components':0})
    lt = transcript_map.get(final_id, {'turns':[],'text':'','word_count':0,'n_components':0})
    rows.append({
        'keydevid':int(r.keydevid), 'companyid':int(r.companyid),
        'companyname':r.companyname, 'country_code':r.country_code,
        'call_date':r.call_date.strftime('%Y-%m-%d'),
        'call_time_utc':safe_str(r.call_time_utc),
        'first_transcriptid':first_id,'first_created_at_utc':str(r.first_created_at_utc),
        'final_transcriptid':final_id,'final_created_at_utc':str(r.final_created_at_utc),
        'n_versions':int(r.n_versions),
        'first_word_count':ft['word_count'],'final_word_count':lt['word_count'],
        'first_n_components':ft['n_components'],'final_n_components':lt['n_components'],
        'first_text':ft['text'],'final_text':lt['text'] if FETCH_FINAL_VERSION else '',
        'first_turns':ft['turns'],'final_turns':lt['turns'] if FETCH_FINAL_VERSION else [],
    })
records = pd.DataFrame(rows)
print('Events reconstructed:',len(records))
display(records[['companyname','call_date','first_word_count','final_word_count','n_versions']].head(20))

Events reconstructed: 5


,companyname,call_date,first_word_count,final_word_count,n_versions
0,JPMorgan Chase & Co.,2025-10-14,9007,9040,4
1,The Coca-Cola Company,2025-10-21,9347,9356,3
2,Caterpillar Inc.,2025-10-29,9171,9149,3
3,Microsoft Corporation,2025-10-29,8698,8658,3
4,Oracle Corporation,2025-12-10,6867,6856,3


## 6. Optional GVKEY link for joining Compustat later

Use an event-date-aware link and prefer the primary mapping. **Do not invent tickers.** Match to returns data using a verified security identifier and date-valid link in the later merge notebook.

In [20]:
gv = db.raw_sql('SELECT companyid, gvkey, startdate, enddate, primaryflag FROM ciq.wrds_gvkey',
                date_cols=['startdate','enddate'])
link = records[['keydevid','companyid','call_date']].copy()
link['call_date'] = pd.to_datetime(link.call_date)
link = link.merge(gv,on='companyid',how='left')
valid = ((link.startdate.isna() | (link.call_date >= link.startdate)) &
         (link.enddate.isna() | (link.call_date <= link.enddate)))
link = link[valid | link.gvkey.isna()].copy()
link['primary_sort'] = link.primaryflag.astype(str).str.lower().isin(['1','true','t','y','yes']).astype(int)
link = link.sort_values(['keydevid','primary_sort','startdate'],ascending=[True,False,False]).drop_duplicates('keydevid')
records = records.merge(link[['keydevid','gvkey']],on='keydevid',how='left',validate='one_to_one')
print('GVKEY linked:',records.gvkey.notna().sum(),'/',len(records))

GVKEY linked: 5 / 5


## 7. Quality control and export

All failed checks are **flagged, not discarded**. In particular, call-date and creation-date fields are distinct; the latter can be missing or delayed. Export metadata to CSV, transcript text and turns to JSONL. Keep raw transcript exports in approved research storage only.

In [21]:
records['missing_first_text'] = records.first_word_count.eq(0)
records['short_first_text'] = records.first_word_count.lt(MIN_WORDS)
records['missing_gvkey'] = records.gvkey.isna()
records['missing_first_created_at'] = records.first_created_at_utc.isin(['NaT','None',''])
records['first_version_created_after_call'] = (
    pd.to_datetime(records.first_created_at_utc,utc=True,errors='coerce').dt.date >
    pd.to_datetime(records.call_date).dt.date)
records['needs_review'] = records[['missing_first_text','short_first_text','missing_first_created_at']].any(axis=1)
assert records.keydevid.is_unique, 'Duplicate event IDs in final output'
assert records.call_date.between(START_DATE,END_DATE).all(), 'Call outside requested date range'

text_fields = ['first_text','final_text','first_turns','final_turns']
metadata = records.drop(columns=text_fields)
metadata.to_csv(OUT/'earnings_events.csv',index=False)
metadata[metadata.needs_review].to_csv(OUT/'earnings_events_exceptions.csv',index=False)
with (OUT/'earnings_event_transcripts.jsonl').open('w',encoding='utf-8') as f:
    for obj in records.to_dict(orient='records'):
        # Normalize NaN / NaT values for valid JSON
        obj = {k:(None if isinstance(v,float) and pd.isna(v) else v) for k,v in obj.items()}
        f.write(json.dumps(obj,ensure_ascii=False,default=str,allow_nan=False)+'\n')
print('Exported:',OUT)
print('Events:',len(metadata),'Review flags:',metadata.needs_review.sum())
display(metadata[['companyname','call_date','gvkey','first_word_count','n_versions','needs_review']].head(30))

Exported: c:\Users\vidhi\OneDrive\Desktop\Project lab\Autumn 26\LLM-based-earnings-surprise_NTAM\data\processed\earnings_events_q4_2025
Events: 5 Review flags: 0


,companyname,call_date,gvkey,first_word_count,n_versions,needs_review
0,JPMorgan Chase & Co.,2025-10-14,002968,9007,4,False
1,The Coca-Cola Company,2025-10-21,003144,9347,3,False
2,Caterpillar Inc.,2025-10-29,002817,9171,3,False
3,Microsoft Corporation,2025-10-29,012141,8698,3,False
4,Oracle Corporation,2025-12-10,012142,6867,3,False


## 8. Research handoff / limitations

**Outputs:**
- `data/processed/earnings_events_q4_2025/earnings_events.csv` — one row per event, metadata and QC flags
- `.../earnings_events_exceptions.csv` — events needing manual review
- `.../earnings_event_transcripts.jsonl` — full text + structured speaker turns, first/final versions

**Before notebook 03:** inspect the QC report; verify company identifiers; confirm the relevant *earnings announcement timestamp* independently; confirm transcript version publication/availability timing. For true point-in-time LLM scores, use only text actually available at prediction time. The LLM notebook should normally start from `first_text`, not `final_text`.

**Scaling:** set `PILOT_ONLY=False` and `MAX_EVENTS=None` after the pilot is validated. Fetching all Q4 events can take substantial time and storage; confirm licensing and team infrastructure first. Close connection after finishing.

In [22]:
db.close()
print('WRDS connection closed')

WRDS connection closed


In [25]:
import pandas as pd

events = pd.read_csv(OUT / "earnings_events.csv")

print("Total earnings events:", len(events))
print("\nColumns:", events.columns.tolist())

display(events.head(10))

print("\nMissing values:")
print(events.isna().sum())

print("\nUnique companies:",
      events["companyid"].nunique())

print("\nUnique earnings events:",
      events["keydevid"].nunique())

Total earnings events: 5

Columns: ['keydevid', 'companyid', 'companyname', 'country_code', 'call_date', 'call_time_utc', 'first_transcriptid', 'first_created_at_utc', 'final_transcriptid', 'final_created_at_utc', 'n_versions', 'first_word_count', 'final_word_count', 'first_n_components', 'final_n_components', 'gvkey', 'missing_first_text', 'short_first_text', 'missing_gvkey', 'missing_first_created_at', 'first_version_created_after_call', 'needs_review']


,keydevid,companyid,companyname,country_code,call_date,call_time_utc,first_transcriptid,first_created_at_utc,final_transcriptid,final_created_at_utc,...,final_word_count,first_n_components,final_n_components,gvkey,missing_first_text,short_first_text,missing_gvkey,missing_first_created_at,first_version_created_after_call,needs_review
0,1880672021,658776,JPMorgan Chase & Co.,US,2025-10-14,12:30:00,3560280,2025-10-14 16:17:46+00:00,3617520,2025-12-10 16:53:33+00:00,...,9040,62,67,2968,False,False,False,False,False,False
1,1962423929,26642,The Coca-Cola Company,US,2025-10-21,12:30:00,3562705,2025-10-21 13:53:59+00:00,3619015,2025-12-11 21:09:17+00:00,...,9356,54,54,3144,False,False,False,False,False,False
2,1958574325,259378,Caterpillar Inc.,US,2025-10-29,12:30:00,3570518,2025-10-29 13:44:54+00:00,3572025,2025-10-30 09:08:54+00:00,...,9149,42,42,2817,False,False,False,False,False,False
3,1964399669,21835,Microsoft Corporation,US,2025-10-29,21:30:00,3571533,2025-10-29 22:46:03+00:00,3576017,2025-11-03 10:58:26+00:00,...,8658,34,33,12141,False,False,False,False,False,False
4,1972492780,22247,Oracle Corporation,US,2025-12-10,22:00:00,3617991,2025-12-10 23:00:12+00:00,3618101,2025-12-11 06:25:58+00:00,...,6856,29,29,12142,False,False,False,False,False,False



Missing values:
keydevid                            0
companyid                           0
companyname                         0
country_code                        0
call_date                           0
call_time_utc                       0
first_transcriptid                  0
first_created_at_utc                0
final_transcriptid                  0
final_created_at_utc                0
n_versions                          0
first_word_count                    0
final_word_count                    0
first_n_components                  0
final_n_components                  0
gvkey                               0
missing_first_text                  0
short_first_text                    0
missing_gvkey                       0
missing_first_created_at            0
first_version_created_after_call    0
needs_review                        0
dtype: int64

Unique companies: 5

Unique earnings events: 5
